In [31]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal
from langchain_openai.chat_models import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
load_dotenv()

True

# Table of Contents 

1. Separate LLMs
2. Creating the chat notes individually
3. Building the subgraph and the main graph

**Both will have their separate LLM models.**

In [32]:
Parent_LLM  =  ChatOpenAI(model="gpt-4.1-mini")
subgraph_LLM = ChatOpenAI(model="gpt-4.1-mini")


**Create the chat nodes**
1)  is that both will inherit from the same chat state. 


In [33]:
class Translate(TypedDict):
    
    query :  str
    english_ans :  str 
    hindi_ans  : str 
    
    

def answer_question(state :  Translate) :
    
    state["english_ans"]  =  Parent_LLM.invoke(state["query"])
    return {"english_ans":state["english_ans"].content}



def translate_text(state : Translate) :
    
    state["hindi_ans"] =  subgraph_LLM.invoke(f"Convert {state['english_ans']} to the Hindi language. ")
    
    return { "hindi_ans" :  state["hindi_ans"].content }

In [34]:
subraph =  StateGraph(Translate)
subraph.add_node("translate",translate_text)


subraph.add_edge(START,"translate")
subraph.add_edge("translate",END)

subgraph =  subraph.compile()



main_graph = StateGraph(Translate)
main_graph.add_node("ans_question",answer_question)
main_graph.add_node("translation",subgraph)


main_graph.add_edge(START,"ans_question")
main_graph.add_edge("ans_question","translation")
main_graph.add_edge("translation",END)


workflow =  main_graph.compile()



In [35]:

result =  workflow.invoke({"query":"what is the value of pi ? "})
print(result)

{'query': 'what is the value of pi ? ', 'english_ans': "The value of pi (π) is approximately 3.14159. It is an irrational number, meaning it has an infinite number of non-repeating decimal places. Pi represents the ratio of a circle's circumference to its diameter.", 'hindi_ans': 'पाई (π) का मान लगभग 3.14159 है। यह एक अपरिमेय संख्या है, जिसका अर्थ है कि इसके दशमलव स्थान अनंत और गैर-पुनरावृत्त होते हैं। पाई किसी वृत्त की परिधि और उसके व्यास के बीच का अनुपात दर्शाता है।'}
